# Fine-tune ViT5 cho Query Rewrite
- **Môi trường**: Kaggle
- **Phần cứng**: 2x T4 GPU (Hỗ trợ DataParallel tự động thông qua HuggingFace Trainer)
- **Model**: VietAI/vit5-base
- **Dataset**: rewrite.csv (1 cột `raw_query`, 1 cột `optimized_query`)

In [ ]:
!pip install -q -U transformers tokenizers && !pip install -q datasets evaluate accelerate rouge_score sentencepiece

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Load secret từ Kaggle Secrets (cần tạo secret tên hf_token trong menu Add-ons -> Secrets)
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")
login(token=hf_token)

In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv('rewrite.csv')
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

In [ ]:
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM

model_name = "VietAI/vit5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    # Thêm prefix vào câu hỏi đầu vào, T5 thường học qua prefix
    inputs = ["rewrite: " + doc for doc in examples["raw_query"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)

    # Chuẩn bị labels cho đầu ra (query đã được tối ưu)
    labels = tokenizer(text_target=examples["optimized_query"], max_length=64, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

In [ ]:
from transformers import DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Data collator sẽ tự động lo việc padding các input và label
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Thay thế ID -100 bằng pad_token_id vì không decode được -100
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="vit5-query-rewriter",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16, # Batch size TRÊN MỖI GPU (2x T4 => Global batch = 32)
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=10,
    predict_with_generate=True, # Cần thiết cho Seq2Seq evaluation
    fp16=True, # T4 support FP16
    push_to_hub=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
# Bắt đầu huấn luyện
trainer.train()

In [ ]:
# Push model và tokenizer lên Hugging Face Hub
trainer.push_to_hub("Đã train xong ViT5 Query Rewrite")